In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)

In [2]:
df = pd.read_csv('/kaggle/input/datasets/ijk037/race-data/race_data.csv')

horses = [
    "Shadowfax", "Iron Duke", "Morningstar", "Red Tide",
    "Gallant Fox", "Blue Streak", "Copper Prince", "Last Chance"
]

In [3]:
mu = df[horses].mean().values
sigma = df[horses].std().values

sigma = np.maximum(sigma, 1e-6)

In [4]:
# Mean and std
mu = df[horses].mean().values
sigma = df[horses].std().values

sigma = np.maximum(sigma, 1e-6)

# Correlation matrix
corr_matrix = df[horses].corr().values

# Covariance matrix
cov_matrix = np.outer(sigma, sigma) * corr_matrix

# Stability fix (VERY IMPORTANT)
cov_matrix += np.eye(len(horses)) * 1e-6

In [5]:
N_SIM = 100000

samples = np.random.multivariate_normal(mean=mu, cov=cov_matrix, size=N_SIM)

# Find winners (min time per row)
winner_idx = np.argmin(samples, axis=1)

win_counts = np.bincount(winner_idx, minlength=len(horses))

# Convert to probabilities
p = {horses[i]: win_counts[i] / N_SIM for i in range(len(horses))}

print("Estimated Probabilities (Correlated):")
for h in horses:
    print(f"{h}: {p[h]:.4f}")

Estimated Probabilities (Correlated):
Shadowfax: 0.3261
Iron Duke: 0.1409
Morningstar: 0.3192
Red Tide: 0.0605
Gallant Fox: 0.0570
Blue Streak: 0.0423
Copper Prince: 0.0305
Last Chance: 0.0235


In [6]:
f = {
    "Shadowfax": 0.08,
    "Iron Duke": 0.09,
    "Morningstar": 0.11,
    "Red Tide": 0.13,
    "Gallant Fox": 0.14,
    "Blue Streak": 0.15,
    "Copper Prince": 0.14,
    "Last Chance": 0.16
}

In [7]:
ev = {}

for h in horses:
    ev[h] = p[h] * (0.85 / f[h]) - 1

print("\nExpected Value:")
for h in horses:
    print(f"{h}: {ev[h]:.4f}")


Expected Value:
Shadowfax: 2.4653
Iron Duke: 0.3308
Morningstar: 1.4663
Red Tide: -0.6046
Gallant Fox: -0.6541
Blue Streak: -0.7605
Copper Prince: -0.8147
Last Chance: -0.8750


In [8]:
bankroll = 10000
fraction = 0.5  # fractional Kelly

bets = {}

for h in horses:
    edge = p[h] * (0.85 / f[h]) - 1
    odds = (0.85 / f[h]) - 1
    
    if edge > 0:
        kelly = edge / odds
        bets[h] = bankroll * fraction * kelly
    else:
        bets[h] = 0

# Normalize if needed
total_bet = sum(bets.values())
if total_bet > bankroll:
    scale = bankroll / total_bet
    for h in bets:
        bets[h] *= scale

In [9]:
print("\nRecommended Stakes (£):")
for h in horses:
    print(f"{h}: £{bets[h]:.2f}")

print(f"\nTotal Bet: £{sum(bets.values()):.2f}")


Recommended Stakes (£):
Shadowfax: £1280.70
Iron Duke: £195.88
Morningstar: £1089.83
Red Tide: £0.00
Gallant Fox: £0.00
Blue Streak: £0.00
Copper Prince: £0.00
Last Chance: £0.00

Total Bet: £2566.40
